# Initialize

In [ ]:
!pip install scikit-learn==1.6.1
!pip install xgboost==3.2.0
!pip install optuna==4.8.0
!pip install openpyxl==3.1.5

from google.colab import drive

drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 4.7 MB/s eta 0:00:00
Mounted at /content/drive


# XGB Regressor

In [ ]:
import json
import os
import re
import hashlib
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

def save_results(rows, path):
    df_res = pd.DataFrame(rows)

    df_res = df_res.drop_duplicates(
        subset=["dataName", "seed","TW", "outer_fold"],
        keep="last"
    )
    df_res.to_excel(path, index=False)

def ensure_categorical_dtype_from_train(train_df, test_df, cat_cols):
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in cat_cols:
        if col in train_df.columns:
            cats = pd.Index(train_df[col].dropna().unique())
            dtype = pd.CategoricalDtype(categories=cats, ordered=False)
            train_df[col] = train_df[col].astype(dtype)
            test_df[col] = test_df[col].astype(dtype)

    return train_df, test_df

for dataName in ["/content/drive/MyDrive/Colab_Backups/Data_3sMean_IDT_updateSaccade1.xlsx"]:

  for rs in [
      261,
      452,
      285,
      966,786,
      1423,
      3789, 5602, 8741, 11905
      ]:

      TW_comb = [[0,x] for x in range(1,31)]
      twType = "combineTW"
      # twType = "singleTW"

      # if rs == 966:
      #    TW_comb = [[0, x] for x in range(15,31)]
      # else:
      #    TW_comb = [[0, x] for x in range(1,31)]

      for TWs in TW_comb:

        print(TWs)

        if len(TWs)>1:

          targetTW = TWs[1]

        else:
          targetTW = TWs[0]

        corrThre = 0.85

        # ---------- Data Setup ----------
        df = pd.read_excel(dataName)

        BASE_DIR = "/content/drive/MyDrive/Colab_Backups/XGBR"
        os.makedirs(BASE_DIR, exist_ok=True)

        results_path = os.path.join(BASE_DIR, f"XGBR_{twType}.xlsx")
        modelDirectory = f"XGB_model_{twType}"


        if os.path.exists(results_path):
              results_df = pd.read_excel(results_path)
              results_rows = results_df.to_dict('records')
              print(f"Resuming from existing file. Found {len(results_rows)} previous runs.")
        else:
              results_df = pd.DataFrame(columns=["seed", "TW"])
              results_rows = []

        # ---- skip if this rs + TW already exists ----
        if not results_df.empty:
              existing = (
                  (results_df["seed"] == rs) &
                  (results_df["TW"] == targetTW)
              )

              if existing.any():
                  print(f"Skipping rs={rs}, targetTW={targetTW} because it already exists.")
                  continue

        # ---- run your modelling code here ----
        print(f"Running rs={rs}, targetTW={targetTW}")



        # ---------- Parameter ----------
        outer_splits, inner_splits, n_trials = 5, 4, 100

        target_col = "relativeRT"

        cat_names = ["Gender", "automation", "eventType"]

        TW_list = TWs

        feature_templates = [
            "Age",
            "License",
            "Mileage",
            "Gender",
            "automation", "eventType",
            "visibleSpeed",
            # "sAmpMean_{TW}", "sVel2Mean_{TW}",
            # "sDuMean_{TW}",

            "pdMean_{TW}", "pdSd_{TW}",
            "blinkNumber_{TW}",
            "avgBlinkDuration_{TW}",

            "fixNumber_{TW}",
            "avgFixDuration_{TW}",

            "fixTE_{TW}", "fixSE_{TW}",
            "FixDisperX_{TW}", "FixDisperY_{TW}",
            "FixDisperE_{TW}",
            "FixMeanPosX_{TW}", "FixMeanPosY_{TW}",

        ]

        feature_cols = []
        for c in feature_templates:
            feature_cols.extend([c.format(TW=tw) for tw in TW_list] if "{TW}" in c else [c])

        cont_names = [c for c in feature_cols if c not in cat_names]

        tww_cols = [f"perValid_{i}" for i in range(31)]

        pervalid_mean = df[tww_cols].mean(axis=1)

        abandon_mask = pervalid_mean < 74.5
        keep_mask = ~abandon_mask

        df = df[keep_mask].copy()

        X = df[feature_cols].copy()
        y = pd.to_numeric(df[target_col], errors="coerce").values.astype(float)
        groups = df["subNumber"].values

        outer_cv = GroupKFold(
            n_splits=outer_splits,
            shuffle=True,
            random_state=rs
        )

        outer_scores = []
        outer_RMSE = []
        oof_pred = np.full_like(y, fill_value=-1)
        oof_fold = np.full(len(y), -1, dtype=int)
        oof_proba = np.full(len(y), np.nan)
        all_fold_best_vars = []

        outer_best_params = []
        outer_R2 = []
        fold_rows = []   # temp storage for this (dataName, rs, TW)

        print(f'len(df)={len(df)}')

        for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups), start=1):


            print(f"\n=== Outer fold {outer_fold}/{outer_splits} ===")

            X_out_train = X.iloc[train_idx].copy()
            y_out_train = y[train_idx]
            groups_out_tr = groups[train_idx]

            X_out_test = X.iloc[test_idx].copy()
            y_out_test = y[test_idx]

            to_drop_fold = []

            for col in cat_names:
                if col in X_out_train.columns:
                    print(f"Fold {outer_fold} - {col} levels in train:",
                          X_out_train[col].unique())


            fold_vars = [c for c in feature_cols if c not in to_drop_fold]
            fold_cont = [c for c in fold_vars if c in cont_names]
            fold_cat = [c for c in fold_vars if c in cat_names]

            print(f"  > [TW-Filter] Dropped {len(to_drop_fold)} smaller windows. {len(fold_vars)} variables remain.")

            current_vars = fold_vars[:]
            best_vars = current_vars

            X_out_train_oh, X_out_test_oh = ensure_categorical_dtype_from_train(
                  X_out_train[best_vars],
                  X_out_test[best_vars],
                  cat_names
              )

            def objective(trial):

                p = {
                    "n_estimators":  200,
                    "max_depth": trial.suggest_int("max_depth", 2, 7),
                    "gamma": trial.suggest_float("gamma", 0.0, 2.0),
                    "learning_rate": trial.suggest_float("learning_rate", 0.025, 0.3),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.25, 1.0),

                    "tree_method": "hist",
                    "random_state": 42,
                    "n_jobs": -1,
                    "enable_categorical": True,
                    "objective": "reg:squarederror",
                }


                cv_opt = GroupKFold(
                    n_splits=inner_splits,
                    shuffle=True,
                    random_state=123
                )
                scores = []


                for t_i, v_i in cv_opt.split(X_out_train_oh, y_out_train, groups_out_tr):

                    X_tr_opt = X_out_train_oh.iloc[t_i].copy()
                    X_val_opt = X_out_train_oh.iloc[v_i].copy()

                    y_tr = y_out_train[t_i]
                    y_val = y_out_train[v_i]

                    m = XGBRegressor(**p)
                    m.fit(X_tr_opt, y_tr, verbose=False)

                    y_pred = m.predict(X_val_opt)

                    scores.append(np.sqrt(mean_squared_error(y_val, y_pred)))

                return float(np.mean(scores))


            sampler = optuna.samplers.TPESampler(seed=42)
            study = optuna.create_study(direction="minimize", sampler=sampler)
            study.optimize(objective, n_trials=n_trials)

            best_trial = study.best_trial

            final_p = {
                **study.best_params,
                "n_estimators": 200,
                "tree_method": "hist",
                "random_state": 42,
                "n_jobs": -1,
                "enable_categorical": True,
                "objective": "reg:squarederror",
            }

            final_model = XGBRegressor(**final_p)
            outer_best_params.append(final_p)

            final_model.fit(X_out_train_oh, y_out_train)
            y_pred = final_model.predict(X_out_test_oh)


            oof_pred[test_idx] = y_pred
            oof_fold[test_idx] = outer_fold


            outer_rmse = np.sqrt(mean_squared_error(y_out_test, y_pred))

            r2   = r2_score(y_out_test, y_pred)

            fold_rows.append({
                "dataName": dataName,
                "seed": rs,
                "TW": float(targetTW),
                "outer_fold": int(outer_fold),
                "RMSE": float(outer_rmse),
                "R2": float(r2),
                "n_test": int(len(test_idx)),
            })

            plt.figure(figsize=(5, 5))
            plt.scatter(y_out_test, y_pred, alpha=0.6)
            outer_R2.append(r2)

            outer_scores.append(outer_rmse)

            print(f"{dataName} | Outer fold {outer_fold}/5 RMSE = {outer_rmse:.4f} R2 = {r2:.4f}")

            lims = [
                min(y_out_test.min(), y_pred.min()),
                max(y_out_test.max(), y_pred.max())
            ]
            plt.plot(lims, lims)
            plt.xlim(lims)
            plt.ylim(lims)

            plt.xlabel("Observed")
            plt.ylabel("Predicted")
            plt.title(
                f"Outer fold {outer_fold}\n"
                f"RMSE = {outer_rmse:.3f}, R² = {r2:.3f}"
            )

            plt.tight_layout()
            plt.show()

            MODEL_DIR = os.path.join(BASE_DIR, modelDirectory)
            os.makedirs(MODEL_DIR, exist_ok=True)

            model_path = os.path.join(MODEL_DIR, f"xgb_seed_{rs:02d}_TW_{targetTW:02d}_outerfold_{outer_fold:02d}.json")
            final_model.get_booster().save_model(model_path)
            print("Saved model:", model_path)



            meta = {
                "seed": rs,
                "TW": targetTW,
                "outer_fold": outer_fold,
                "best_params": final_p,
                "best_score_inner_cv": study.best_value,
                "selected_features": best_vars
            }

            meta_path = model_path.replace(".json", "_meta.json")

            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=4)

            print("Saved metadata:", meta_path)


        # ---------- Final reporting ----------
        print(f"\n===== FINAL RESULTS SUMMARY =====")
        print(f"Mean RMSE (Outer CV): {np.mean(outer_scores):.4f} (+/- {np.std(outer_scores):.4f})")
        print(f"Mean R2 (Outer CV): {np.mean(outer_R2):.4f} (+/- {np.std(outer_R2):.4f})")


        results_rows.extend(fold_rows)


        summary_row = {
            "dataName": dataName,
            "seed": rs,
            "TW": float(targetTW),
            "outer_fold": "SUMMARY",
            "RMSE": float(np.mean(outer_scores)),
            "R2": float(np.mean(outer_R2)),
            "std_RMSE": float(np.std(outer_scores)),
            "std_R2": float(np.std(outer_R2)),
            "n_samples": int(len(y)),
        }
        results_rows.append(summary_row)

        save_results(results_rows, results_path)
        print(f"Updated results: {results_path}")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import math
def ReactionPredicMetric_HazardFeatures(subno,velocityThre,TW, foldName):
    def Blink(data):
        blink = 0
        blinkDuration = np.nan
        sdblinkDuration = np.nan
        mdblinkDuration = np.nan
        if len(data[data['Blink'] != 0]) != 0:
            blink = data.loc[data['Blink'] != 0, 'Blink'].nunique()
            blinkcode = data[data['Blink'] != 0]['Blink'].unique()
            blinkdu = []
            for blinkID in blinkcode:
                blinkdu.append(data[data['Blink'] == blinkID]['TimeStamp'].iloc[-1] -
                               data[data['Blink'] == blinkID]['TimeStamp'].iloc[0])
            blinkDuration = np.mean(blinkdu) / 10000
            sdblinkDuration = np.std(blinkdu) / 10000
            mdblinkDuration = np.median(blinkdu) / 10000
        return blink, blinkDuration, sdblinkDuration,mdblinkDuration
    def CalculateSEntropy(data, xcol, ycol, grid):

        data = data[
            (data[xcol] >= -25) & (data[xcol] <= 25)
            ]
        data = data[
            (data[ycol] >= -25) & (data[ycol] <= 25)
            ]
        gridWid = grid
        gridHei = grid
        x = data[xcol]
        y = data[ycol]

        xbins = np.arange(-25, 25 + gridWid, gridWid)
        ybins = np.arange(-25, 25 + gridHei, gridHei)
        H, _, _ = np.histogram2d(x, y, bins=[xbins, ybins])
        p = H.ravel() / H.sum()
        p = p[p > 0]
        entropy = -np.sum(p * np.log2(p))
        print(f"Entropy = {entropy:.4f} bits")
        return entropy
    def CalculateTEntropy(data, xcol, ycol, grid):
        # Assume `data` is in time order already.
        idx_orig = np.arange(len(data))  # original positions

        # reliable_mask = (data['GazeDirectionUnreliable'] != 1) if xcol != 'fixHeading' else np.ones(len(data), bool)

        # 2) Scale
        xv = (data[xcol]).to_numpy()
        yv = (data[ycol]).to_numpy()

        # 3) In-bounds masks
        in_x = (xv >= -25) & (xv <= 25)
        in_y = (yv >= -25) & (yv <= 25)

        # valid = reliable_mask & in_x & in_y
        valid =  in_x & in_y

        # 5) Binning for ALL samples (we'll ignore invalids later)
        gridWid = grid
        gridHei = grid
        xbins = np.arange(-25, 25 + gridWid, gridWid)
        ybins = np.arange(-25, 25 + gridHei, gridHei)
        x_idx = np.digitize(xv, xbins) - 1
        y_idx = np.digitize(yv, ybins) - 1

        n_x = len(xbins) - 1
        n_y = len(ybins) - 1

        x_idx = np.clip(x_idx, 0, n_x - 1)
        y_idx = np.clip(y_idx, 0, n_y - 1)

        # AOI index for all; set invalids to -1 so we can skip them cleanly
        AOI_index = x_idx * n_y + y_idx
        AOI_index[~valid] = -1

        # 6) Count transitions ONLY when two samples are consecutive in the ORIGINAL sequence
        n_aoi = n_x * n_y
        trans_counts = np.zeros((n_aoi, n_aoi), dtype=int)

        for i in range(len(AOI_index) - 1):
            a, b = AOI_index[i], AOI_index[i + 1]
            # require both valid and consecutive (i and i+1 are consecutive by construction)
            if a >= 0 and b >= 0 and a != b:
                trans_counts[a, b] += 1

        # 7) Row-normalize
        row_sums = trans_counts.sum(axis=1, keepdims=True)
        P = np.divide(trans_counts, row_sums, out=np.zeros_like(trans_counts, dtype=float), where=row_sums != 0)

        # 8) Stationary visit probs over valid samples only
        visits = AOI_index[AOI_index >= 0]
        unique, counts = np.unique(visits, return_counts=True)
        pi = np.zeros(n_aoi)
        if counts.sum() > 0:
            pi[unique] = counts / counts.sum()

        # 9) Transition entropy
        H = 0.0
        for i in range(n_aoi):
            for j in range(n_aoi):
                p_ij = P[i, j]
                if p_ij > 0:
                    H -= pi[i] * p_ij * np.log2(p_ij)
        return H
    def FixationDispersion(data):
        data = data[data['location'].str.contains('windscreen', case=False, na=False)]
        data['fixWorldX']=data['fixHeading']
        data['fixWorldY'] = data['fixPitch']
        data['fixWorldE'] = np.sqrt(data['fixWorldX'] ** 2 + data['fixWorldY'] ** 2)
        return data['fixWorldX'].std(), data['fixWorldY'].std(), data['fixWorldE'].std(),data['fixWorldX'].mean(), data['fixWorldY'].mean(), data['fixWorldE'].mean(),data['fixWorldX'].median(), data['fixWorldY'].median(), data['fixWorldE'].median()
    def GetSaccadeFeatures(data):
        keep_mask = []

        return (
            data['duration'].mean(),
            data['duration'].std(),

            data['amplitude_deg'].mean(),
            data['amplitude_deg'].std(),

            data['peak_vel_deg_s'].mean(),
            data['peak_vel_deg_s'].std(),

            data['mean_vel_deg_s'].mean(),
            data['mean_vel_deg_s'].std(),

            data['duration'].median(),
            data['amplitude_deg'].median(),
            data['peak_vel_deg_s'].median(),
            data['mean_vel_deg_s'].median()
        )
    def median_filter_series(s, window=5):
        return s.rolling(window=window, center=True, min_periods=window).median()
    counterBalance = pd.read_csv("/content/drive/MyDrive/HiDrive4/Material/study4_counterbalancing.csv", delimiter=',', skiprows=1)
    scheduleAction = pd.read_csv("/content/drive/MyDrive/HiDrive4/Material/HAROLD_study3_schedule_actions.csv", delimiter=',', skiprows=1)
    eyeEvent = pd.read_excel("/content/drive/MyDrive/HiDrive4/Material/PhysicalRT_31Jul_0.001_2_0.75.xlsx")

    # pre-index for faster lookup
    counterBalance['sub'] = counterBalance['sub'].astype(int)
    counterBalance['run'] = counterBalance['run'].astype(int)
    counter_idx = counterBalance.set_index(['sub', 'run'])

    eyeEvent = eyeEvent.dropna(subset=['subNumber', 'runNumber', 'section']).copy()
    eyeEvent['subNumber'] = eyeEvent['subNumber'].astype(int)
    eyeEvent['runNumber'] = eyeEvent['runNumber'].astype(int)
    eyeEvent['section'] = eyeEvent['section'].astype(int)
    eye_idx = eyeEvent.set_index(['subNumber', 'runNumber', 'section'])

    scheduleAction = scheduleAction[
        scheduleAction['section'].between(3, 8)
    ].copy()

    rows = []

    skip_pairs = {(1, 2), (27, 3)}
    skip_subs = {2, 18, 28, 37}

    for subNumber in subno:
        for runNumber in [1, 2, 3, 4]:
            if (subNumber, runNumber) in skip_pairs or subNumber in skip_subs:
                continue

            cb_sub = subNumber % 24
            if cb_sub == 0:
                cb_sub = 24

            if (cb_sub, runNumber) not in counter_idx.index:
                continue

            cb_row = counter_idx.loc[(cb_sub, runNumber)]
            automation = cb_row['automation']
            nback = cb_row['nback']

            simData = pd.read_csv(
                "/content/drive/MyDrive/HiDrive4/HiDrive4_SimOnly/HiDrive4_S{}_R{}_signals100000.txt".format(subNumber,
                                                                                                   runNumber),
                delimiter='\t')
            eyeData = pd.read_excel('/content/drive/MyDrive/HiDrive4/All_Filtered Eye/s{}_r{}.xlsx'.format(subNumber, runNumber))

            blink_mask = (eyeData['Blink'] != 0)
            combine_mask = blink_mask

            PD_masked = pd.Series(
                np.where(~combine_mask, eyeData['PupilDiameter'], np.nan),
                index=eyeData.index
            )

            interp = pd.Series(PD_masked).interpolate(method='linear')
            PD_smoothed = median_filter_series(interp, window=3)
            eyeData['PD_Smooth'] = PD_smoothed

            saccadeData = pd.read_excel(
                '/content/drive/MyDrive/HiDrive4/Fixation_and_Saccade/All_Saccade_{}/s{}_r{}.xlsx'.format(foldName, subNumber, runNumber))
            saccadeData = saccadeData[saccadeData['mean_vel_deg_s']>30]

            fixData = pd.read_excel(
                '/content/drive/MyDrive/HiDrive4/Fixation_and_Saccade/All_Fixation_{}/s{}_r{}.xlsx'.format(foldName, subNumber, runNumber))

            fixData['fixStartTime']*=1000
            fixData['fixEndTime'] *= 1000
            fixData['fixWorldX'] *= -100
            fixData['fixWorldY'] *= 100
            halfBound = 1.36529135

            run_schedule = scheduleAction[
                (scheduleAction['sub'] == subNumber) &
                (scheduleAction['run'] == runNumber) &
                (scheduleAction['tilenum'] == 2)
                ]

            for section in [3, 4, 5, 6, 7, 8]:

                sec_schedule = run_schedule[run_schedule['section'] == section]
                if len(sec_schedule) == 0:
                    continue
                eventType = sec_schedule['eventType'].iloc[0]

                if ("NON" in eventType):
                    continue

                print("s{}r{}section{}event{}".format(subNumber, runNumber, section, eventType))

                key = (subNumber, runNumber, section)
                if key not in eye_idx.index:
                    continue

                eye_row = eye_idx.loc[key]
                matTime = eye_row['critical_simTime']
                rt = eye_row['relativeRT']
                rtSim = rt + matTime

                critical_rows = simData[
                    (simData['roadID_major'] == section) &
                    (simData['roadID_minor'] == 2) &
                    (simData['distance'] >= 150)
                    ]
                if len(critical_rows) == 0:
                    continue

                criticalSim = critical_rows['simTime'].iloc[0]
                criticalSpeed = critical_rows['speed'].iloc[0]

                # __________________________________________________
                # Ambient Measures

                # if "LEFT" in eventType:
                #     extra = 0.65/0.75
                # elif "RIGHT" in eventType:
                #     extra = 0.65/1.5
                # elif "ONCOMING" in eventType:
                #     extra = 0
                epochStart = criticalSim - TW
                epochEnd = min(matTime, rtSim) if TW == 0 else criticalSim

                epochSim = simData[(simData['simTime'] >= epochStart) & (simData['simTime'] <= epochEnd)]
                epochSim_valid = epochSim[epochSim['smartEyeFrameNumber'] != 0]
                epochDuration = epochEnd-epochStart
                # visibleSpeed = simData[simData['simTime']>=criticalSim]['speed'].iloc[0]

                if len(epochSim_valid) == 0:
                    rows.append({
                        'subNumber': subNumber,
                        'runNumber': runNumber,
                        'automation': automation,
                        'nback': nback,
                        'section': section,
                        'eventType': eventType,
                        'epochDuration':epochDuration,
                    })
                    continue

                epochStartFrame = epochSim_valid['smartEyeFrameNumber'].iloc[0]
                epochEndFrame = epochSim_valid['smartEyeFrameNumber'].iloc[-1]

                start_eye = eyeData[eyeData['FrameNumber'] >= epochStartFrame]
                end_eye = eyeData[eyeData['FrameNumber'] >= epochEndFrame]
                if len(start_eye) == 0 or len(end_eye) == 0:
                    continue

                epochStartEyeTime = start_eye['TimeStamp'].iloc[0] / 10000
                epochEndEyeTime = end_eye['TimeStamp'].iloc[0] / 10000

                epochSaccade = saccadeData[
                    (saccadeData['frameStart'] >= epochStartFrame) &
                    (saccadeData['frameStart'] <= epochEndFrame)
                    ]
                epochFix = fixData[
                    (fixData['frameEnd'] >= epochStartFrame) &
                    (fixData['frameStart'] <= epochEndFrame)
                    ].copy()
                epochEye = eyeData[
                    (eyeData['FrameNumber'] >= epochStartFrame) &
                    (eyeData['FrameNumber'] <= epochEndFrame)
                    ]

                saccNumber = len(epochSaccade)
                if saccNumber == 0:
                    sDuMean = np.nan
                    sDuSd = np.nan
                    sAmpMean = np.nan
                    sAmpSd = np.nan
                    sVelMean = np.nan
                    sVelSd = np.nan
                    sVel2Mean = np.nan
                    sVel2Sd = np.nan
                    sDuMd = np.nan
                    sAmpMd = np.nan
                    sVelMd =np.nan
                    sVel2Md = np.nan
                else:
                    sDuMean, sDuSd, sAmpMean, sAmpSd, sVelMean, sVelSd, sVel2Mean, sVel2Sd, sDuMd, sAmpMd, sVelMd,sVel2Md = GetSaccadeFeatures(
                        epochSaccade.copy())


                pdMean = epochEye['PD_Smooth'].mean()
                pdSd = epochEye['PD_Smooth'].std()
                pdMd = epochEye['PD_Smooth'].median()

                blinkNumber, blinkDuration, sdBlinkDuration, mdBlinkDuration = Blink(epochEye.copy())

                fixNumber = len(epochFix)
                if fixNumber == 0:
                    rows.append({
                        'subNumber': subNumber,
                        'runNumber': runNumber,
                        'automation': automation,
                        'nback': nback,
                        'section': section,
                        'eventType': eventType,
                        f'epochDuration_{TW}':epochDuration,
                        f'sDuMean_{TW}': sDuMean,
                        f'sDuSd_{TW}': sDuSd,
                        f'sAmpMean_{TW}': sAmpMean,
                        f'sAmpSd_{TW}': sAmpSd,
                        f'sVelMean_{TW}': sVelMean,
                        f'sVelSd_{TW}': sVelSd,
                        f'sVel2Mean_{TW}': sVel2Mean,
                        f'sVel2Sd_{TW}': sVel2Sd,
                        f'sDuMd_{TW}': sDuMd,
                        f'sAmpMd_{TW}': sAmpMd,
                        f'sVelMd_{TW}': sVelMd,
                        f'sVel2Md_{TW}': sVel2Md,
                        f'pdMean_{TW}': pdMean,
                        f'pdSd_{TW}': pdSd,
                        f'pdMd_{TW}': pdMd,
                        f'blinkNumber_{TW}': blinkNumber,
                        f'avgBlinkDuration_{TW}': blinkDuration,
                        f'sdBlinkDuration_{TW}': sdBlinkDuration,
                        f'mdBlinkDuration_{TW}': mdBlinkDuration,
                    })
                    continue

                epochFix['fixEndTime'] = epochFix['fixEndTime'].clip(upper=epochEndEyeTime)
                epochFix['fixStartTime'] = epochFix['fixStartTime'].clip(lower=epochStartEyeTime)
                epochFix['adjFixDuration'] = epochFix['fixEndTime'] - epochFix['fixStartTime']
                epochFix = epochFix[epochFix['adjFixDuration'] >= 100]

                FixDisperX, FixDisperY, FixDisperE, FixMeanPosX, FixMeanPosY, FixMeanPosE, FixMedPosX, FixMedPosY, FixMedPosE = FixationDispersion(
                    epochFix.copy())

                fixSE_2 = CalculateSEntropy(epochFix.copy(), 'fixHeading', 'fixPitch', 1)
                fixTE_2 = CalculateTEntropy(epochFix.copy(), 'fixHeading', 'fixPitch', 1)

                fixDuration_md = epochFix['adjFixDuration'].median()
                fixDuration_mean = epochFix['adjFixDuration'].mean()
                fixDuration_sd = epochFix['adjFixDuration'].std()

                base_row = {
                    'subNumber': subNumber,
                    'runNumber': runNumber,
                    'automation': automation,
                    'nback': nback,
                    'section': section,
                    'eventType': eventType,

                }

                epoch_metrics = {
                    'epochDuration': epochDuration,
                    'saccNumber': saccNumber,
                    'sDuMean': sDuMean,
                    'sDuSd': sDuSd,
                    'sDuMd': sDuMd,
                    'sAmpMean': sAmpMean,
                    'sAmpSd': sAmpSd,
                    'sAmpMd': sAmpMd,
                    'sVelMean': sVelMean,
                    'sVelSd': sVelSd,
                    'sVel2Mean': sVel2Mean,
                    'sVel2Sd': sVel2Sd,
                    'sVelMd': sVelMd,
                    'sVel2Md': sVel2Md,

                    'pdMean': pdMean,
                    'pdSd': pdSd,
                    'pdMd': pdMd,
                    'blinkNumber': blinkNumber,
                    'avgBlinkDuration': blinkDuration,
                    'sdBlinkDuration': sdBlinkDuration,
                    'mdBlinkDuration': mdBlinkDuration,
                    'fixNumber': fixNumber,
                    'avgFixDuration': fixDuration_mean,
                    'sdFixDuration': fixDuration_sd,
                    'mdFixDuration': fixDuration_md,
                    'fixTE': fixTE_2,
                    'fixSE': fixSE_2,

                    'FixDisperX': FixDisperX,
                    'FixDisperY': FixDisperY,
                    'FixDisperE': FixDisperE,

                    'FixMeanPosX': FixMeanPosX,
                    'FixMeanPosY': FixMeanPosY,
                    'FixMeanPosE': FixMeanPosE,

                    'FixMdPosX': FixMedPosX,
                    'FixMdPosY': FixMedPosY,
                    'FixMdPosE': FixMedPosE,

                }
                metrics_tw = {
                    f'{k}_{TW}': v
                    for k, v in epoch_metrics.items()
                }

                rows.append({**base_row, **metrics_tw})

    dfResult = pd.DataFrame(rows)

    save_dir = f"/content/drive/MyDrive/HiDrive4/ETFeatures/Updated_Saccade2/Model_{foldName}"

    os.makedirs(save_dir, exist_ok=True)

    save_path = f"{save_dir}/ETFeatures_{TW}TW_{velocityThre}Vel.xlsx"

    dfResult.to_excel(save_path, index=False)

In [ ]:
subno = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
             25, 27, 28, 29, 30, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48]


for targetTW in range(0,31):
        ReactionPredicMetric_HazardFeatures(subno, TW=targetTW, velocityThre=30, foldName = '3sMean_IDT')

s1r1section4eventEVENT_MAT_ONCOMING
Entropy = 2.0000 bits
s1r1section6eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s1r1section8eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s1r3section3eventEVENT_MAT_LEFT_VRU
Entropy = 1.5000 bits
s1r3section5eventEVENT_MAT_RIGHT_VRU
Entropy = 2.4194 bits
s1r3section6eventEVENT_MAT_ONCOMING
Entropy = 0.8113 bits
s1r4section3eventEVENT_MAT_ONCOMING
Entropy = 1.0000 bits
s1r4section6eventEVENT_MAT_RIGHT_VRU
Entropy = 1.9183 bits
s1r4section8eventEVENT_MAT_LEFT_VRU
Entropy = 1.3710 bits
s3r1section4eventEVENT_MAT_RIGHT_VRU
Entropy = 1.4591 bits
s3r1section6eventEVENT_MAT_LEFT_VRU
Entropy = 1.2516 bits
s3r1section7eventEVENT_MAT_ONCOMING
Entropy = 2.8454 bits
s3r2section4eventEVENT_MAT_LEFT_VRU
Entropy = 2.4464 bits
s3r2section7eventEVENT_MAT_RIGHT_VRU
Entropy = 2.5503 bits
s3r2section8eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s3r3section3eventEVENT_MAT_ONCOMING
Entropy = 1.7527 bits
s3r3section5eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s3r3sec

/tmp/ipykernel_10724/1853869038.py:39: RuntimeWarning: invalid value encountered in divide
  p = H.ravel() / H.sum()


s5r2section8eventEVENT_MAT_ONCOMING
Entropy = 1.0000 bits
s5r3section4eventEVENT_MAT_LEFT_VRU
Entropy = 1.5850 bits
s5r3section5eventEVENT_MAT_ONCOMING
Entropy = 1.0000 bits
s5r3section7eventEVENT_MAT_RIGHT_VRU
Entropy = -0.0000 bits
s5r4section4eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s5r4section6eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s5r4section7eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s6r1section4eventEVENT_MAT_LEFT_VRU
Entropy = 2.0000 bits
s6r1section5eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s6r1section7eventEVENT_MAT_RIGHT_VRU
Entropy = 1.5850 bits
s6r2section5eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s6r2section7eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s6r2section8eventEVENT_MAT_ONCOMING
Entropy = 0.9183 bits
s6r3section3eventEVENT_MAT_RIGHT_VRU
Entropy = 0.9183 bits
s6r3section5eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s6r3section6eventEVENT_MAT_ONCOMING
Entropy = 2.0000 bits
s6r4section4eventEVENT_MAT_ONCOMING
Entropy = 0.9183 bits
s6r4s

/tmp/ipykernel_10724/1853869038.py:39: RuntimeWarning: invalid value encountered in divide
  p = H.ravel() / H.sum()


s16r3section8eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s16r4section3eventEVENT_MAT_LEFT_VRU
Entropy = 1.5850 bits
s16r4section7eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s16r4section8eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s17r1section3eventEVENT_MAT_RIGHT_VRU
Entropy = -0.0000 bits
s17r1section4eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s17r1section7eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s17r2section5eventEVENT_MAT_ONCOMING
Entropy = 1.0000 bits
s17r2section7eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s17r2section8eventEVENT_MAT_RIGHT_VRU
Entropy = 1.5850 bits
s17r3section3eventEVENT_MAT_ONCOMING
Entropy = 0.9183 bits
s17r3section6eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s17r3section7eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s17r4section3eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s17r4section4eventEVENT_MAT_LEFT_VRU
Entropy = 2.0000 bits
s17r4section7eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s19r1section3eventEVENT_MAT_RIGHT_VRU
Entrop

/tmp/ipykernel_10724/1853869038.py:39: RuntimeWarning: invalid value encountered in divide
  p = H.ravel() / H.sum()


s21r1section4eventEVENT_MAT_RIGHT_VRU
s21r1section6eventEVENT_MAT_ONCOMING
Entropy = 1.5850 bits
s21r1section8eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s21r2section4eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s21r2section6eventEVENT_MAT_ONCOMING
Entropy = 1.5850 bits
s21r2section8eventEVENT_MAT_LEFT_VRU
Entropy = 1.5850 bits
s21r3section4eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s21r3section5eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s21r3section8eventEVENT_MAT_ONCOMING
s21r4section3eventEVENT_MAT_ONCOMING
Entropy = 0.9183 bits
s21r4section4eventEVENT_MAT_LEFT_VRU
Entropy = 1.5000 bits
s21r4section5eventEVENT_MAT_RIGHT_VRU
Entropy = -0.0000 bits
s22r1section4eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s22r1section5eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s22r1section6eventEVENT_MAT_ONCOMING
Entropy = 0.9183 bits
s22r2section6eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s22r2section7eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s22r2section8eventEVENT_MAT_RIG

/tmp/ipykernel_10724/1853869038.py:39: RuntimeWarning: invalid value encountered in divide
  p = H.ravel() / H.sum()


s35r2section4eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s35r2section6eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s35r3section4eventEVENT_MAT_RIGHT_VRU
Entropy = 1.5850 bits
s35r3section5eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s35r3section6eventEVENT_MAT_ONCOMING
Entropy = 1.5850 bits
s35r4section3eventEVENT_MAT_LEFT_VRU
Entropy = -0.0000 bits
s35r4section4eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s35r4section5eventEVENT_MAT_RIGHT_VRU
Entropy = -0.0000 bits
s36r1section4eventEVENT_MAT_ONCOMING
Entropy = 1.0000 bits
s36r1section5eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s36r1section7eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s36r2section4eventEVENT_MAT_RIGHT_VRU
Entropy = 1.0000 bits
s36r2section6eventEVENT_MAT_ONCOMING
Entropy = 1.0000 bits
s36r2section7eventEVENT_MAT_LEFT_VRU
Entropy = 1.0000 bits
s36r3section5eventEVENT_MAT_ONCOMING
Entropy = -0.0000 bits
s36r3section6eventEVENT_MAT_RIGHT_VRU
Entropy = 0.9183 bits
s36r3section7eventEVENT_MAT_LEFT_VRU
Entropy 

/tmp/ipykernel_10724/1853869038.py:39: RuntimeWarning: invalid value encountered in divide
  p = H.ravel() / H.sum()


Streaming output truncated to the last 5000 lines.
s48r1section5eventEVENT_MAT_RIGHT_VRU
Entropy = 2.5547 bits
s48r2section4eventEVENT_MAT_LEFT_VRU
Entropy = 2.0679 bits
s48r2section5eventEVENT_MAT_RIGHT_VRU
Entropy = 3.2829 bits
s48r2section8eventEVENT_MAT_ONCOMING
Entropy = 2.2846 bits
s48r3section6eventEVENT_MAT_ONCOMING
Entropy = 4.0739 bits
s48r3section7eventEVENT_MAT_LEFT_VRU
Entropy = 4.2148 bits
s48r3section8eventEVENT_MAT_RIGHT_VRU
Entropy = 4.0892 bits
s48r4section4eventEVENT_MAT_LEFT_VRU
Entropy = 3.8275 bits
s48r4section5eventEVENT_MAT_ONCOMING
Entropy = 2.3408 bits
s48r4section6eventEVENT_MAT_RIGHT_VRU
Entropy = 3.8019 bits
s1r1section4eventEVENT_MAT_ONCOMING
Entropy = 3.5475 bits
s1r1section6eventEVENT_MAT_LEFT_VRU
Entropy = 4.1957 bits
s1r1section8eventEVENT_MAT_RIGHT_VRU
Entropy = 3.8597 bits
s1r3section3eventEVENT_MAT_LEFT_VRU
Entropy = 2.9457 bits
s1r3section5eventEVENT_MAT_RIGHT_VRU
Entropy = 2.9814 bits
s1r3section6eventEVENT_MAT_ONCOMING
Entropy = 3.8074 bits
s1r4s